# Closed-Thought LLM — Evaluation on Colab A100

Run KV recurrence experiments (exp7b) on GSM8K and ARC benchmarks.

**Requirements:** Colab Pro with A100 runtime (40GB VRAM).

In [ ]:
# Step 1: Check GPU
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 2: Clone repo and install dependencies
from getpass import getpass
token = getpass("GitHub token (leave blank if repo is public): ")
if token:
    !git clone https://{token}@github.com/PatrickJaiin/closed-thought-llm.git
else:
    !git clone https://github.com/PatrickJaiin/closed-thought-llm.git

%cd closed-thought-llm
!pip install -q torch transformers accelerate datasets peft bitsandbytes

In [ ]:
# Step 3: Verify model loads correctly
from config import DEVICE, DTYPE, MODEL_NAME, LOAD_IN_4BIT
print(f"Device: {DEVICE}")
print(f"Dtype: {DTYPE}")
print(f"Model: {MODEL_NAME}")
print(f"4-bit: {LOAD_IN_4BIT}")

In [ ]:
# Step 4: Quick sanity check — run eval on 20 built-in prompts
!python experiments/exp7b_kv_generation.py \
    --eval-only \
    --no-flash-attn \
    --configs "KV7B-0,KV7B-B,KV7B-D,KV7B-G90"

In [ ]:
# Step 5: GSM8K benchmark (subset=200 for ~30min, full=1319 for ~3hrs)
!python experiments/exp7b_kv_generation.py \
    --benchmark gsm8k \
    --subset 200 \
    --no-flash-attn \
    --configs "KV7B-0,KV7B-A,KV7B-B,KV7B-D,KV7B-G90,KV7B-FTO"

In [ ]:
# Step 6: ARC benchmark
!python experiments/exp7b_kv_generation.py \
    --benchmark arc \
    --subset 200 \
    --no-flash-attn \
    --configs "KV7B-0,KV7B-A,KV7B-B,KV7B-D,KV7B-G90,KV7B-FTO"

In [ ]:
# Step 7: View results
import json

for name in ["exp7b_eval_results", "exp7b_gsm8k_results", "exp7b_arc_results"]:
    path = f"results/{name}.json"
    try:
        with open(path) as f:
            data = json.load(f)
        print(f"\n{'='*50}")
        print(f"{name}")
        print(f"{'='*50}")
        for k, v in data.items():
            if k == "metadata":
                continue
            if isinstance(v, dict) and "accuracy" in v:
                print(f"  {k}: {v['accuracy']:.1%} ({v.get('correct', '?')}/{v.get('total', '?')})")
    except FileNotFoundError:
        print(f"{name}: not found (skipped)")

In [ ]:
# Step 8: Download results to local machine
from google.colab import files
!zip -r results.zip results/
files.download('results.zip')